# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hicham1236/Intern-Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task Type:** Binary Classification applied as Scoring/Ranking.
**Why:** We are predicting a binary state (is this page decaying? Yes/No). However, the business needs a prioritized list, not just categories. By using the predicted probabilities (scoring), we can rank the pages from highest risk to lowest, giving the content team a manageable queue.



In [1]:
import os, sys, subprocess
import pandas as pd

# Environment setup to ensure data loads smoothly in Colab
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"We have {df.shape[0]} pages. A simple classification is not enough; we must rank them to find the top priority ones.")

We have 30000 pages. A simple classification is not enough; we must rank them to find the top priority ones.


## 2. Target or proxy

**Target:** The target is a derived proxy label is_declining, based on the defined rule trend_direction == 'down'.

**Why a proxy?:** The true business goal is finding "pages that will regain traffic if rewritten." Since we lack historical rewrite success data in this starter set, we use the observed downward trend as a proxy indicator of decay.

In [2]:
# Creating the target proxy based on the observed rule
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Distribution of our proxy target:")
print(df["is_declining"].value_counts(normalize=True).round(3))

Distribution of our proxy target:
is_declining
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

**Metric:** Precision@K (e.g., Precision@50).

**Defending it:** The content team has limited bandwidth. Recall (finding all decaying pages) is useless if the team can only update 50 pages a month. Precision@50 ensures that out of the top 50 recommendations, the highest possible percentage are genuinely decaying pages worth the team's effort.

In [3]:
# If we picked 50 pages at random, our baseline Precision@50 would be the global decline rate.
baseline_precision = df["is_declining"].mean()
print(f"Baseline Precision@50 (random guessing): {baseline_precision:.3f}")
print("A 'good' model must significantly beat this baseline (e.g., finding actual signal deep in the list).")

Baseline Precision@50 (random guessing): 0.542
A 'good' model must significantly beat this baseline (e.g., finding actual signal deep in the list).


## 4. The unit of analysis, as a real dataframe

**Unit of Analysis:** One row = One specific web page (URL). The dataset aggregates performance metrics (impressions, age, positioning) at the page level.

In [4]:
print(f"Unit of analysis: {df.shape[0]} rows (pages) x {df.shape[1]} columns (features/labels)")

# Showing the slice
display_cols = ["content_age_days", "impressions_90d", "trend_direction", "is_declining"]
display(df[display_cols].head())

Unit of analysis: 30000 rows (pages) x 45 columns (features/labels)


,content_age_days,impressions_90d,trend_direction,is_declining
0,187,3803,down,1
1,445,15320,down,1
2,141,12581,down,1
3,463,11751,stable,0
4,263,19140,down,1


## 5. Why ML beats a fixed rule here

A fixed hand-written rule is too rigid. If we just say "update old pages," we create a huge backlog of false positives (old pages that are still performing perfectly well). ML captures non-linear interactions (e.g., how age, click-through rate, and position interact) to score risk accurately, turning a massive list into a sharp, prioritized queue.

In [5]:
# Showing why a simple 'age' rule is messy
old_pages = df[df["content_age_days"] > 180]
decline_rate_old = old_pages["is_declining"].mean()

print(f"If we use a fixed rule (Age > 180 days), we flag {len(old_pages)} pages.")
print(f"But {100 - (decline_rate_old * 100):.1f}% of them are NOT declining. ML is needed to find the real signals without wasting editorial time.")

If we use a fixed rule (Age > 180 days), we flag 17728 pages.
But 51.7% of them are NOT declining. ML is needed to find the real signals without wasting editorial time.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.